# Tutorial: Using the Content Generator Module

This notebook demonstrates the `content_generator.py` module, which is designed to generate synthetic conversational data, including full conversation transcripts, individual turns, agent personas, and associated metadata. It leverages large language models (LLMs) to create realistic and varied content for training and testing conversational AI systems, such as contact center AI solutions.

## Setup: Installation and Mocking Internal LibrariesThe `content_generator.py` module relies on `strenum` and Google Cloud's `google-generativeai` for LLM interaction. It also imports from internal `conidk.core` and `conidk.wrapper.vertex` modules. For this demonstration, we will install the publicly available libraries and provide mock implementations for the `conidk` components to allow the `Generator` class to be instantiated and its methods called in a Colab environment without requiring the full `conidk` library setup.Additionally, the `Generator` class reads JSON schema files from a `utils/schemas` directory and a default agents file from `utils/defaults`. We will create these directories and populate them with dummy JSON files within the notebook for a self-contained demonstration.

In [ ]:
# Install necessary public libraries
!pip install -q google-generativeai strenum

import sys
import json
import random
import datetime
import enum
import os
import pathlib
from typing import List, Optional

# --- Mocking `conidk` internal libraries ---

# Mock conidk.core.base
class MockBaseAuth:
    def __init__(self):
        pass

class MockBaseConfig:
    def __init__(self):
        pass

# Create dummy conidk.core module and its base submodule
sys.modules['conidk'] = type('module', (object,), {})()
sys.modules['conidk.core'] = type('module', (object,), {'base': type('module', (object,), {'Auth': MockBaseAuth, 'Config': MockBaseConfig})()})
sys.modules['conidk.core.base'] = sys.modules['conidk.core'].base


# Mock conidk.wrapper.vertex
class MockGeminiModels(enum.Enum):
    # These models should align with those expected by content_generator.py
    GEMINI_1_5_PRO_PREVIEW_0514 = "gemini-1.5-pro-preview-0514"
    GEMINI_2_5_PRO = "gemini-2.5-pro" # Placeholder for a likely future model, used in default profile
    GEMINI_1_0_PRO = "gemini-1.0-pro"

class MockMimeTypes(enum.Enum):
    APPLICATION_JSON = "application/json"
    TEXT_PLAIN = "text/plain"

class MockVertexGenerator:
    """Mock Vertex AI Generator that attempts to use actual `google-generativeai`
    if configured, otherwise falls back to dummy responses."""
    def __init__(self, project_id: str, location: str):
        self.project_id = project_id
        self.location = location
        self.version = MockGeminiModels.GEMINI_1_0_PRO # Default model, can be updated by content_generator.py
        print(f"Mock VertexGenerator initialized for project: {project_id} in {location}")

        self._model = None
        try:
            import google.generativeai as genai
            genai.configure(project=project_id) # Configure with project_id for ADC
            # Initialize with a default model, it will be updated by content_generator.py
            self._model = genai.GenerativeModel(MockGeminiModels.GEMINI_1_0_PRO.value)
            print(f"  (Mock VertexGenerator using actual genai.GenerativeModel for project: {project_id})")
        except Exception as e:
            print(f"  (Could not initialize actual google.generativeai.GenerativeModel for mock: {type(e).__name__}: {e}. Will use dummy responses.)")

    def content(
        self,
        prompt: str,
        output_schema: dict,
        output_mime_type: MockMimeTypes,
        temperature: float,
        top_p: float,
        top_k: int,
    ) -> str:
        print(f"\n--- MOCK GENERATOR CALL (Model: {self.version.value}) ---")
        print(f"  Prompt (first 150 chars): {prompt[:150]}...")
        
        # Attempt to use the actual LLM if available and configured
        if self._model:
            # Update the model if self.version has changed in the calling `Generator`
            target_model_name = self.version.value
            if not self._model.model_name.endswith(target_model_name): 
                 try:
                     import google.generativeai as genai
                     self._model = genai.GenerativeModel(target_model_name)
                     print(f"  (Switched actual genai.GenerativeModel to: {target_model_name})")
                 except Exception as e:
                     print(f"  (Failed to switch model to {target_model_name}: {e}. Keeping current or falling back.)")
                     self._model = None # Disable LLM if model switch fails

            if self._model: # Re-check if model is still valid after potential switch
                try:
                    # Instruct the model to strictly follow the schema
                    full_prompt = (
                        f"{prompt}\n\n"
                        f"Please generate a JSON object conforming strictly to the following JSON schema:\n"
                        f"```json\n{json.dumps(output_schema, indent=2)}\n```\n"
                        "Do not include any text or explanations outside of the JSON object. "
                        "Ensure the JSON is perfectly valid and can be parsed directly."
                    )
                    
                    response = self._model.generate_content(
                        full_prompt,
                        generation_config=genai.GenerationConfig(
                            temperature=temperature,
                            top_p=top_p,
                            top_k=top_k,
                            response_mime_type=output_mime_type.value # This helps enforce JSON output
                        )
                    )
                    generated_text = response.text
                    
                    # Attempt to parse and re-dump to ensure validity
                    parsed = json.loads(generated_text)
                    print("  (LLM returned valid JSON output)")
                    return json.dumps(parsed)
                except Exception as e:
                    print(f"  Error using actual LLM ({type(e).__name__}: {e}). Falling back to dummy response.")
        
        # Dummy response logic if LLM failed or not available
        print("  (Using dummy JSON response)")
        schema_title = output_schema.get("title", "").lower()
        now = datetime.datetime.now().isoformat()
        
        if "conversation" in schema_title:
            return json.dumps({
                "conversation_id": f"mock_convo_{random.randint(1000, 9999)}",
                "turns": [
                    {"speaker": "CUSTOMER", "text": "Hello, my service isn't working.", "timestamp": now, "sentiment": "negative"},
                    {"speaker": "AGENT", "text": "I understand. Let me help you with that.", "timestamp": now, "sentiment": "neutral"}
                ]
            })
        elif "turn" in schema_title:
            is_customer_turn = "CUSTOMER" in prompt or "Start the conversation" in prompt
            text_options = {
                "CUSTOMER": [
                    "My internet has been really slow lately, what's going on?",
                    "I want to upgrade my entertainment package.",
                    "Why was I charged an extra fee this month?",
                    "QUIT" # Allow dummy to quit
                ],
                "AGENT": [
                    "I can definitely look into that for you.",
                    "Let me check your account details.",
                    "Please bear with me while I access your information."
                ]
            }
            speaker = "CUSTOMER" if is_customer_turn else "AGENT"
            text = random.choice(text_options[speaker])
            sentiment = "negative" if any(s in text for s in ["slow", "extra fee", "charged", "frustrated"]) else "neutral"
            return json.dumps({
                "speaker": speaker,
                "text": text,
                "timestamp": now,
                "sentiment": sentiment
            })
        elif "agent" in schema_title:
            return json.dumps({
                "agents": [
                    {"agent_id": f"{random.randint(100000, 999999)}", "name": "Sophia", "team": "Team A", "sentiment_focus": "positive", "traits": ["empathetic", "active listener"]},
                    {"agent_id": f"{random.randint(100000, 999999)}", "name": "Liam", "team": "Team B", "sentiment_focus": "neutral", "traits": ["analytical", "problem solver"]}
                ]
            })
        else:
            return json.dumps({"mock_response": "Schema not recognized, returning generic mock.", "prompt_excerpt": prompt[:50]})

# Create dummy conidk.wrapper module and its vertex submodule
sys.modules['conidk.wrapper'] = type('module', (object,), {'vertex': type('module', (object,), {'Generator': MockVertexGenerator, 'GeminiModels': MockGeminiModels, 'MimeTypes': MockMimeTypes})()})
sys.modules['conidk.wrapper.vertex'] = sys.modules['conidk.wrapper'].vertex

print("Mock `conidk` modules and `google-generativeai` setup complete.")

## Create Schema Files and Default Agents FileThe `content_generator.py` module relies on several JSON schema files and a default agents file to define the structure of its outputs. We will create these required files in the local filesystem.

In [ ]:
# Create directories for schemas and defaults
os.makedirs('utils/schemas', exist_ok=True)
os.makedirs('utils/defaults', exist_ok=True)
print("Created directories: utils/schemas and utils/defaults")

In [ ]:
%%writefile utils/schemas/conversation.json
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Conversation",
  "description": "Schema for a full conversation transcript.",
  "type": "object",
  "required": ["conversation_id", "turns"],
  "properties": {
    "conversation_id": {
      "type": "string",
      "description": "Unique identifier for the conversation."
    },
    "turns": {
      "type": "array",
      "description": "List of conversational turns.",
      "items": {
        "type": "object",
        "required": ["speaker", "text", "timestamp", "sentiment"],
        "properties": {
          "speaker": {
            "type": "string",
            "enum": ["CUSTOMER", "AGENT"],
            "description": "The speaker of the turn."
          },
          "text": {
            "type": "string",
            "description": "The text content of the turn."
          },
          "timestamp": {
            "type": "string",
            "format": "date-time",
            "description": "Timestamp of the turn."
          },
          "sentiment": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"],
            "description": "Sentiment of the turn."
          }
        }
      }
    }
  }
}

In [ ]:
%%writefile utils/schemas/turn.json
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Turn",
  "description": "Schema for a single turn in a conversation.",
  "type": "object",
  "required": ["speaker", "text", "timestamp", "sentiment"],
  "properties": {
    "speaker": {
      "type": "string",
      "enum": ["CUSTOMER", "AGENT"],
      "description": "The speaker of the turn."
    },
    "text": {
      "type": "string",
      "description": "The text content of the turn. Output 'QUIT' if the conversation should end."
    },
    "timestamp": {
      "type": "string",
      "format": "date-time",
      "description": "Timestamp of the turn."
    },
    "sentiment": {
      "type": "string",
      "enum": ["positive", "neutral", "negative"],
      "description": "Sentiment of the turn."
    }
  }
}

In [ ]:
%%writefile utils/schemas/agent.json
{
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "Agent",
  "description": "Schema for a list of agents.",
  "type": "object",
  "required": ["agents"],
  "properties": {
    "agents": {
      "type": "array",
      "description": "List of agents.",
      "items": {
        "type": "object",
        "required": ["agent_id", "name", "team", "sentiment_focus", "traits"],
        "properties": {
          "agent_id": {
            "type": "string",
            "description": "Unique identifier for the agent."
          },
          "name": {
            "type": "string",
            "description": "Name of the agent."
          },
          "team": {
            "type": "string",
            "description": "Team the agent belongs to."
          },
          "sentiment_focus": {
            "type": "string",
            "enum": ["positive", "neutral", "negative"],
            "description": "Primary sentiment focus for the agent's interactions."
          },
          "traits": {
            "type": "array",
            "items": { "type": "string" },
            "description": "Personality traits of the agent."
          }
        }
      }
    }
  }
}

In [ ]:
%%writefile utils/defaults/agents.json
{
  "agents": [
    {
      "agent_id": "900001",
      "name": "Default Agent A",
      "team": "Default Team X",
      "sentiment_focus": "neutral",
      "traits": ["calm", "efficient"]
    },
    {
      "agent_id": "900002",
      "name": "Default Agent B",
      "team": "Default Team Y",
      "sentiment_focus": "positive",
      "traits": ["friendly", "proactive"]
    }
  ]
}

## The `content_generator.py` Module SourceHere is the full source code for `content_generator.py`. We embed it directly into the notebook for self-containment.

In [ ]:
%%writefile content_generator.py
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Module to generate content for the ingestion pipeline"""

import json
import pathlib
import random
import datetime
from typing import List, Optional

from strenum import StrEnum

from conidk.core import base
from conidk.wrapper import vertex

_DEFAULT_SENTIMENT_DISTRIBUTION = {
    "positive": "70%",
    "neutral": "20%",
    "negative": "10%",
}

_DEFAULT_TRAITS = [
    "amiable",
    "cordial",
    "gracious",
    "approachable",
    "sincere",
    "candid",
    "trustworthy",
    "articulate",
    "expressive",
    "considerate",
    "supportive",
    "cooperative",
    "reserved",
    "quiet",
    "introspective",
    "analytical",
    "practical",
    "stoic",
    "formal",
    "unassuming",
    "individualistic",
    "skeptical",
]

class AgentTeamDistribution(StrEnum):
    """Enum for agent team distribution options."""

    EVENLY = "evenly"
    RANDOM = "random"
    UNBALANCED = "unbalanced"

class Themes(StrEnum):
    """Enum for different themes."""

    TELCO = "Telecommunications"
    ENTERTAINMENT = "Entertainment"

class SentimentJourneys(StrEnum):
    """Enum for different sentiment journeys."""

    HIGH_TO_LOW = "High to Low"
    LOW_TO_HIGH = "Low to High"
    NEUTRAL_TO_LOW = "Neutral to Low"
    NEUTRAL_TO_HIGH = "Neutral to High"
    HIGH_TO_NEUTRAL = "High to Neutral"
    LOW_TO_NEUTRAL = "Low to Neutral"
    STABLE = "Stable with no change"

class Generator:
    """Generates specific sets of content using a large language model.

    This class provides methods to generate synthetic conversational data, including
    conversation transcripts, agent personas, and other metadata. It uses a large
    language model to create realistic and varied content for training and testing
    conversational AI systems.

    Attributes:
        project_id: The Google Cloud project ID.
        location: The location of the Vertex AI endpoint (e.g., 'us-central1').
        auth: An authentication object.
        config: A configuration object.
        generator: An instance of the vertex.Generator class.
    """

    def __init__(
        self,
        project_id: str,
        location: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ):
        """Initializes the content generator.

        Args:
            project_id: The Google Cloud project ID.
            location: The location of the Vertex AI endpoint (e.g., 'us-central1').
            auth: An authentication object.
            config: A configuration object.
        """

        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.project_id = project_id
        self.location = location

        self.generator = vertex.Generator(
            project_id=self.project_id,
            location=self.location,
        )

    def _assign_probabilities(
        self,
        range_long_convo: list,
        range_bad_sentiment: list,
        range_bad_performance: list,
    ):
        """Assign a value based on the probabilities

        Args:
            range_long_convo: expectes a list with two values (min, max)
            range_bad_sentiment: expectes a list with two values (min, max)
            range_bad_performance: expectes a list with two values (min, max)
        """
        long_conversation = False
        bad_sentiment = False
        bad_performance = False

        probability_long_convo = random.uniform(
            range_long_convo[0], range_long_convo[1]
        )
        probability_bad_sentiment = random.uniform(
            range_bad_sentiment[0],
            range_bad_sentiment[1]
        )
        probability_bad_performance = random.uniform(
            range_bad_performance[0],
            range_bad_performance[1]
        )

        if random.uniform(0, 1) < probability_long_convo:
            long_conversation = True

        if random.uniform(0, 1) < probability_bad_sentiment:
            bad_sentiment = True

        if random.uniform(0, 1) < probability_bad_performance:
            bad_performance = True

        return long_conversation, bad_sentiment, bad_performance

    def _set_llm_parameters(
        self,
        range_temperature: list,
        range_topp: list,
        range_topk: list,
    ):
        """Set LLM parameters

        Args:
            range_temperature: expectes a list with two values (min, max)
            range_topp: expectes a list with two values (min, max)
            range_topk: expectes a list with two values (min, max)
        """
        temperature = random.uniform(
            range_temperature[0],
            range_temperature[1]
        )
        topp = random.uniform(
            range_topp[0],
            range_topp[1]
        )
        topk = random.randint(
            int(range_topk[0]),
            int(range_topk[1])
        )
        return temperature, topp, topk

    def _set_prompt_parts(
        self,
        parameters: dict
    ) -> dict[str, str]:
        prompt_parts = {}
        today = datetime.date.today().strftime("%A, %B %d, %Y")

        prompt_parts['language'] = "".join((
            "\n--- LANGUAGE ---"
            "Make sure that the conversation is in ANY of"
            f"the following languages {parameters['language']}"
        ))
        prompt_parts['turn_conclusion'] = "".join((
            "\n--- CONCLUSION ---"
            "If you believe the conversation has completed. just output: QUIT"
        ))
        prompt_parts['turn_sentiment'] = "".join((
            "\n--- SENTIMENT ---"
            f"The conversation's sentiment should follow a {parameters['sentiment_journeys']}"
        ))
        prompt_parts['turn_persona'] = "".join((
            "\n--- PERSONA ---"
            "Always have in mind that you are a CUSTOMER, answering a virtual agent,"
            "so you need to always have customer behaviors"
            "Never repeat something from the context, always generate"
            "a new answer from the CUSTOMER perspective"
            "Always remember, you are a CUSTOMER"
        ))
        prompt_parts['turn_start'] =  "".join((
            f"You are a customer at {parameters['company_name']} and"
            f"you're contact about {parameters['topics']}"
            "Start the conversation with a custom welcome message,"
            "introduce yourself and your problem/issue which has to be"
            f" related to {parameters['topics']} or reason"
            f" to contact {parameters['company_name']}"
        ))
        prompt_parts['context'] = "".join((
            "\n--- CONTEXT ---"
            f"The contact center is in the {parameters['theme']} industry."
            f"The company name is {parameters['company_name']}."
            f"The conversation topic is: {parameters['topics']}."
        ))
        prompt_parts['instructions'] = "".join((
            "\n--- INSTRUCTIONS ---"
            "Make sure that the timestamps are from today's date,"
            f"month and specially YEAR: {today}"
            "Ensure the dialogue is natural and plausible, with realistic phrasing and flow."
        ))
        prompt_parts['output_format'] = "".join((
             "\n--- OUTPUT FORMAT ---"
             "The final output must be a JSON object that strictly conforms"
             "to the provided schema. Do not include any text or explanations"
             "outside of the JSON object."
        ))
        prompt_parts['additional_context'] = "".join((
            "\n--- ADDITIONAL CONTEXT---"
            f"{parameters['hint']}"
        ))
        prompt_parts['bad_performance'] = ""
        prompt_parts['turn_long_conversation'] = ""
        prompt_parts['turn_bad_sentiment'] = ""

        prompt_parts['bad_sentiment'] = "".join((
            f"The conversation's sentiment should follow a {parameters['sentiment_journeys']}"
            "sentiment journey."
        ))
        prompt_parts['long_conversation'] = "".join((
            "The conversation should be of average length,"
            "around 15-25 turns."
        ))
        if parameters['long_conversation']:
            prompt_parts['long_conversation'] = "".join(
                "The conversation must be long, consisting of at least 40 turns."
            )
            prompt_parts['turn_long_conversation'] = "".join(
                "The CUSTOMER must prolong the conversation"
                "with additional questions"
            )
        if parameters['bad_sentiment']:
            prompt_parts['bad_sentiment'] = "".join(
                "The CUSTOMER must display negative sentiment. They should exhibit traits"
                "like frustration, impatience, or confusion."
            )
            prompt_parts['turn_bad_sentiment'] = "".join(
                "The CUSTOMER must display negative sentiment. They should"
                "exhibit traits like frustration, impatience, or confusion."
            )
        if parameters['bad_performance']:
            prompt_parts['bad_performance'] = "".join(
                "The AGENT must perform poorly. They should demonstrate"
                "a lack of knowledge, empathy, or efficiency"
                "(e.g., giving incorrect information, being dismissive, or taking long pauses)."
            )
        return prompt_parts

    def create_parameters(
        self,
        generation_profile: Optional[dict] = None,
        randomize_select: Optional[List] = None,
    ) -> dict:
        """Create a set of random parameters for generating conversations.

        The method arguments serve as the upper bounds for the random generation.

        Args:
            generation_profile: the json provided from the UI
            randomize_select: A list of items to randomly select from.

        Returns:
            A dictionary of all parameters required to generate a conversation or a turn.
        """
        ## Creates default generation profile for non-ui users
        if generation_profile is None:
            generation_profile = {
                "theme": [Themes.ENTERTAINMENT.value], # Use enum value
                "model": MockGeminiModels.GEMINI_2_5_PRO.value, # Use enum value
                "language": ["en-US"],
                "topics": [],
                "probabilities": {
                    "long_conversation": [0.2, 0.4],
                    "bad_sentiment": [0.2, 0.4],
                    "bad_performance": [0.2, 0.4],
                },
                "sentiment_journeys": [SentimentJourneys.NEUTRAL_TO_HIGH.value], # Use enum value
                "temperature": [0.8, 1], 
                "topk": [30, 40], 
                "topp": [0.9, 1], 
                "prompt_hint": [""],
                "company_name": "Cymbal Entertainment"
            }


        randomized_item = random.choice(randomize_select) if randomize_select else None
        conversation_theme = Themes(random.choice(generation_profile['theme']))
        sentiment_journeys = SentimentJourneys(
            random.choice(generation_profile['sentiment_journeys'])
        )
        topic = "Generic"
        if len(generation_profile['topics']) != 0:
            topic = random.choice(generation_profile['topics'])
        company_name = generation_profile.get("company_name", f"Cymbal {conversation_theme}")

        long_conversation , bad_sentiment, bad_performance = self._assign_probabilities(
            generation_profile['probabilities']['long_conversation'],
            generation_profile['probabilities']['bad_sentiment'],
            generation_profile['probabilities']['bad_performance']
        )
        temperature, topp, topk = self._set_llm_parameters(
            generation_profile['temperature'],
            generation_profile['topp'],
            generation_profile['topk']
        )

        return {
            "sentiment_journeys": sentiment_journeys.value,
            "company_name": company_name,
            "bad_performance": bad_performance,
            "long_conversation": long_conversation,
            "bad_sentiment": bad_sentiment,
            "randomize_select": randomized_item,
            "theme": conversation_theme.value,
            "topics": topic,
            "topp": topp,
            "topk": topk,
            "language": generation_profile['language'],
            "temperature": temperature,
            "model": generation_profile['model'], 
            "hint": generation_profile['prompt_hint'][0] if generation_profile['prompt_hint'] else ""
        }

    def create_conversation(
        self,
        parameters: Optional[dict] = None,
    ) -> str:
        """Creates a conversation transcript based on a set of parameters.

        Args:
            parameters: A dictionary of parameters to guide conversation generation.
                Expected keys include 'theme', 'sentiment_journey', 'long_conversation', etc.

        Returns:
            A JSON string representing the generated conversation.
        """
        if parameters is None:
            parameters = self.create_parameters()

        prompt_parts: list[str] = []
        schema_path = pathlib.Path(__file__).parent / "utils/schemas/conversation.json"
        with open(schema_path, "r", encoding="utf-8") as f:
            conversation_schema = json.load(f)

        # Construct prompt parts
        parts = self._set_prompt_parts(parameters=parameters)
        prompt_parts.append(parts['context'])
        prompt_parts.append(parts['instructions'])
        prompt_parts.append(parts['language'])
        prompt_parts.append(parts['long_conversation'])
        prompt_parts.append(parts['bad_sentiment'])
        prompt_parts.append(parts['bad_performance'])
        prompt_parts.append(parts['output_format'])
        prompt_parts.append(parts['additional_context'])

        # Construct the prompt
        prompt = "\n".join(prompt_parts)
        # Update the generator's model version based on parameters
        self.generator.version = vertex.GeminiModels[parameters['model'].replace('-', '_').upper().replace('.', '_')]

        return self.generator.content(
            prompt=prompt,
            output_schema=conversation_schema,
            output_mime_type=vertex.MimeTypes.APPLICATION_JSON,
            temperature=parameters['temperature'],
            top_p=parameters['topp'],
            top_k=parameters['topk']
        )

    def create_turn(
        self,
        parameters: Optional[dict] = None,
        conversation_history: Optional[list] = None,
    ) -> str:
        """Generate a turn of a conversation.

        Args:
            conversation_history: A list of previous turns in the conversation.
            parameters: A dictionary of parameters to guide turn generation.

        Returns:
            A JSON string representing the generated turn.
        """
        if parameters is None:
            parameters = self.create_parameters()

        prompt_parts: list[str] = []
        parts = self._set_prompt_parts(parameters=parameters)

        schema_path = pathlib.Path(__file__).parent / "utils/schemas/turn.json"
        with open(schema_path, "r", encoding="utf-8") as f:
            turn_schema = json.load(f)

        prompt_parts.append(parts['language'])
        if conversation_history is None or len(conversation_history) == 0:
            prompt_parts.append(parts['turn_start'])

        else:
            prompt_parts.append(
                "based on the context below"
                "\n--- CONTEXT ---"
                f"{conversation_history}"
            )
            prompt_parts.append(parts['turn_persona'])
            prompt_parts.append(parts['turn_sentiment'])
            prompt_parts.append(parts['turn_conclusion'])

        prompt_parts.append(parts['turn_bad_sentiment'])
        prompt_parts.append(parts['turn_long_conversation'])
        prompt_parts.append(parts['output_format'])

        prompt = "\n".join(prompt_parts)
        # Update the generator's model version based on parameters
        self.generator.version = vertex.GeminiModels[parameters['model'].replace('-', '_').upper().replace('.', '_')]

        return self.generator.content(
            prompt=prompt,
            output_schema=turn_schema,
            output_mime_type=vertex.MimeTypes.APPLICATION_JSON,
            temperature=parameters['temperature'],
            top_p=parameters['topp'],
            top_k=parameters['topk']
        )

    def create_agents(
        self,
        number_of_agents: Optional[int] = 80,
        number_of_teams: Optional[int] = 8,
        agent_team_distribution: Optional[AgentTeamDistribution] = None,
        sentiment_distribution: Optional[dict] = None,
        traits: Optional[list] = None,
    ) -> dict:
        """Generate a list of agents with metadata information.

        This method creates a list of agents with details like agent ID, name, team,
        and personality traits.

        Args:
            number_of_agents: The number of agents to generate.
            number_of_teams: The number of teams to distribute the agents into.
            agent_team_distribution: The distribution strategy for agents among teams.
            sentiment_distribution: A dictionary defining the sentiment distribution for agents.
            traits: A list of possible personality traits for the agents.

        Returns:
            A dictionary containing the list of generated agents.
        """
        if agent_team_distribution is None:
            agent_team_distribution = AgentTeamDistribution(AgentTeamDistribution.EVENLY)

        if traits is None:
            traits = _DEFAULT_TRAITS
        if sentiment_distribution is None:
            sentiment_distribution = _DEFAULT_SENTIMENT_DISTRIBUTION

        schema_path = pathlib.Path(__file__).parent / "utils/schemas/agent.json"
        with open(schema_path, "r", encoding="utf-8") as f:
            agent_schema = json.load(f)

        prompt = f"""
            Generate a list of {number_of_agents} agents.
            Distribute them into {number_of_teams} teams with a distribution of
            '{agent_team_distribution}'. Make sure that agent names are realistcs
            and the team names are creative (for a contact center).
            The agents should have a sentiment distribution of {sentiment_distribution}.
            The possible traits for the agents are: {traits}, note that each agent should
            have at least four traits. Please provide the output in JSON format conforming
            to the provided schema.
        """
        # For create_agents, we can fix the model or add a parameter
        self.generator.version = vertex.GeminiModels.GEMINI_1_5_PRO_PREVIEW_0514

        content_str = self.generator.content(
            prompt=prompt,
            output_schema=agent_schema,
            output_mime_type=vertex.MimeTypes.APPLICATION_JSON,
            temperature=0.8, # Fixed parameters for agent generation for simplicity
            top_p=0.9,
            top_k=40
        )
        data = json.loads(content_str)

        if "agents" in data and isinstance(data.get("agents"), list):
            # Ensure unique agent_ids if not already present from LLM
            for agent in data["agents"]:
                if "agent_id" not in agent:
                    agent["agent_id"] = str(random.randint(100000, 999999))

        return data

    def create_metadata(
        self,
        parameters: Optional[dict] = None,
        agents: Optional[dict] = None,
    ) -> dict:
        """Generate the required metadata for ingestion into Conversational Insights.

        Args:
            parameters: A dictionary of parameters to guide metadata generation.
            agents: A dictionary containing the list of agents.
        Returns:
            A dictionary containing the generated metadata.
        """
        agents_list = agents
        theme = None
        if agents_list is None:
            schema_path = pathlib.Path(__file__).parent / "utils/defaults/agents.json"
            with open(schema_path, "r", encoding="utf-8") as f:
                agents_list = json.load(f)

        if parameters:
            if "theme" in parameters:
                theme = parameters["theme"]

        if agents_list is None or not agents_list.get("agents"):
            raise RuntimeError("Agent list doesn't exist or is empty")

        number_of_agents = len(agents_list["agents"])
        agent_to_chose = random.randint(0, number_of_agents - 1)
        score = random.randint(1, 5)

        return {
            "customer_satisfaction_rating": score,
            "agent_info": [agents_list["agents"][agent_to_chose]],
            "labels": {"theme": theme, "type":"Synthetic"}
        }


## Import and Initialize the Generator

In [ ]:
# Import the Generator class and its associated Enums
from content_generator import Generator, Themes, SentimentJourneys, AgentTeamDistribution

# --- REPLACE WITH YOUR GOOGLE CLOUD PROJECT ID AND LOCATION ---
# These are crucial for the underlying LLM calls if using actual `google-generativeai`
PROJECT_ID = "YOUR_PROJECT_ID"  # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"} (e.g., 'us-central1', 'europe-west1')
# --- END REPLACE ---

# Initialize the Generator
# For a real application, ensure your environment is authenticated (e.g., `gcloud auth application-default login`)
generator = Generator(project_id=PROJECT_ID, location=LOCATION)

print(f"Content Generator initialized for project: {PROJECT_ID} in {LOCATION}")
print("Note: If `google-generativeai` failed to initialize in the mock, methods will return dummy data.")

## Exploring Enumerations (`Themes`, `SentimentJourneys`, `AgentTeamDistribution`)

In [ ]:
print("Available Themes:")
for theme in Themes:
    print(f"- {theme.name}: {theme.value}")

print("\nAvailable Sentiment Journeys:")
for journey in SentimentJourneys:
    print(f"- {journey.name}: {journey.value}")

print("\nAvailable Agent Team Distributions:")
for dist in AgentTeamDistribution:
    print(f"- {dist.name}: {dist.value}")

## 1. `create_parameters()`: Generating Content ParametersThis method creates a dictionary of randomized parameters that guide the content generation process, such as theme, language, topics, LLM settings, and probabilistic flags for conversation characteristics (e.g., long conversation, bad sentiment).

In [ ]:
print("### Creating Default Parameters ###")
default_params = generator.create_parameters()
print(json.dumps(default_params, indent=2))

print("\n### Creating Custom Parameters with a Generation Profile ###")
custom_generation_profile = {
    "theme": [Themes.TELCO.value], # Use a specific theme
    "model": MockGeminiModels.GEMINI_1_5_PRO_PREVIEW_0514.value, # Specify LLM model
    "language": ["en-US", "es-ES"],
    "topics": ["Billing inquiry", "Technical support for internet outage"],
    "probabilities": {
        "long_conversation": [0.7, 0.9], # High chance of long conversation
        "bad_sentiment": [0.6, 0.8],    # High chance of bad sentiment
        "bad_performance": [0.1, 0.3],  # Low chance of bad agent performance
    },
    "sentiment_journeys": [SentimentJourneys.LOW_TO_HIGH.value], # Customer starts negative, ends positive
    "temperature": [0.7, 0.9], # LLM creativity
    "topk": [20, 30], # LLM token sampling
    "topp": [0.8, 0.9], # LLM token sampling
    "prompt_hint": ["The customer is very particular about data privacy and is calling from California."],
    "company_name": "TelcoConnect Inc."
}
custom_params = generator.create_parameters(generation_profile=custom_generation_profile)
print(json.dumps(custom_params, indent=2))

## 2. `create_agents()`: Generating Agent PersonasThis method generates a list of synthetic agents, each with an ID, name, team, sentiment focus, and personality traits. This data can be used to populate agent databases for simulation or analysis.

In [ ]:
print("### Generating Agents ###")
generated_agents = generator.create_agents(
    number_of_agents=3,
    number_of_teams=2,
    agent_team_distribution=AgentTeamDistribution.RANDOM,
    traits=["friendly", "patient", "knowledgeable", "empathetic", "efficient", "calm"]
)
print(json.dumps(generated_agents, indent=2))

# You can save this to a file for later use
# with open("generated_agents.json", "w") as f:
#     json.dump(generated_agents, f, indent=2)

## 3. `create_conversation()`: Generating a Full Conversation TranscriptThis method generates an entire conversation transcript between a customer and an agent based on the provided parameters. The output is a structured JSON object, making it easy to parse and use.

In [ ]:
print("### Generating a Full Conversation ###")

# We'll use the custom parameters generated earlier, which specify a Telecommunications theme
# and a customer starting with low sentiment moving to high.
conversation_params = generator.create_parameters(generation_profile={
    "theme": [Themes.TELCO.value],
    "model": MockGeminiModels.GEMINI_1_5_PRO_PREVIEW_0514.value,
    "language": ["en-US"],
    "topics": ["Billing dispute for international call"],
    "probabilities": {"long_conversation": [0.5, 0.7], "bad_sentiment": [0.8, 1.0], "bad_performance": [0.0, 0.1]},
    "sentiment_journeys": [SentimentJourneys.LOW_TO_HIGH.value],
    "temperature": [0.8, 0.8],
    "topk": [30, 30],
    "topp": [0.9, 0.9],
    "prompt_hint": ["The customer is initially very angry but calms down as the agent resolves the issue."],
    "company_name": "GlobalTel"
})

generated_conversation_json = generator.create_conversation(parameters=conversation_params)

print("\nGenerated Conversation (JSON string):\n")
print(generated_conversation_json)

# Parse and display for better readability
generated_conversation_data = json.loads(generated_conversation_json)
print("\n### Parsed Conversation: ###")
print(json.dumps(generated_conversation_data, indent=2))

## 4. `create_turn()`: Generating Individual Conversation TurnsThis method allows for generating a conversation turn by turn, providing a `conversation_history` to guide the LLM. This is useful for interactive simulations or for extending existing conversations.

In [ ]:
print("### Generating Conversation Turns Iteratively ###")

turn_params = generator.create_parameters(generation_profile={
    "theme": [Themes.ENTERTAINMENT.value],
    "model": MockGeminiModels.GEMINI_1_5_PRO_PREVIEW_0514.value,
    "language": ["en-US"],
    "topics": ["Subscription cancellation", "Trouble streaming content"],
    "sentiment_journeys": [SentimentJourneys.HIGH_TO_LOW.value], # Customer starts positive, becomes negative
    "temperature": [0.8, 0.8], 
    "topk": [30, 30],
    "topp": [0.9, 0.9],
    "prompt_hint": ["The customer is polite but becomes increasingly frustrated with technical issues."],
    "company_name": "StreamFlix"
})

conversation_history = []
max_turns = 6
print("Starting interactive conversation...")

for i in range(max_turns):
    print(f"\n--- Turn {i+1} ---")
    turn_json = generator.create_turn(
        parameters=turn_params,
        conversation_history=conversation_history
    )
    turn_data = json.loads(turn_json)
    
    speaker = turn_data.get('speaker', 'UNKNOWN')
    text = turn_data.get('text', 'No text.')
    timestamp = turn_data.get('timestamp', '')

    print(f"[{timestamp}] {speaker}: {text}")

    conversation_history.append(turn_data)

    # Check for a 'QUIT' signal from the LLM
    if text.strip().upper() == 'QUIT':
        print("Conversation concluded by LLM.")
        break

print("\n### Full Conversation History (from turns): ###")
print(json.dumps(conversation_history, indent=2))

## 5. `create_metadata()`: Generating Conversation MetadataThis method generates auxiliary metadata for a conversation, such as customer satisfaction ratings, agent information, and labels. This data is crucial for populating conversational analytics platforms like Google Cloud's Contact Center AI Insights.

In [ ]:
print("### Generating Metadata ###")

# Use the agents generated earlier and some theme parameters
metadata_params = {"theme": Themes.ENTERTAINMENT.value}
generated_metadata = generator.create_metadata(parameters=metadata_params, agents=generated_agents)

print(json.dumps(generated_metadata, indent=2))

print("\n### Generating Metadata with default agents (if none provided) ###")
metadata_default_agents = generator.create_metadata(parameters=metadata_params)
print(json.dumps(metadata_default_agents, indent=2))

## ConclusionThis tutorial demonstrated how to use the `content_generator.py` module to create various types of synthetic conversational data:
- `create_parameters()`: To generate randomized configuration settings for content generation.
- `create_agents()`: To produce synthetic agent profiles with traits and team assignments.
- `create_conversation()`: To generate complete, multi-turn conversation transcripts based on defined scenarios.
- `create_turn()`: To generate individual turns, allowing for iterative conversation building.
- `create_metadata()`: To generate supplementary data like customer satisfaction ratings and agent details.By leveraging these functions, developers and researchers can efficiently generate large volumes of diverse and realistic conversational data, which is invaluable for developing, testing, and fine-tuning conversational AI models and applications.